# 02 · RFM Customer Segmentation

**Goal:** group customers by how recently they bought (**R**ecency), how often (**F**requency) and how much they spent (**M**onetary), so each group can get a different action.

**Source:** `gold.fact_sales` (orders with a valid date)

**Method:** each metric gets a quartile score from 1 to 4, and the three scores are combined into five business segments.

**Output:** segment table, segment summary and two charts in `output/`

## 1. Setup

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import config

sns.set_style("whitegrid")
os.makedirs("output", exist_ok=True)

## 2. Load sales and set the snapshot date

The snapshot date is the last order date plus one day, so recency is counted from the end of the data and not from today.

In [19]:
engine = create_engine(config.get_conn_url())
df_sales = pd.read_sql(
    """ select customer_key, order_number, order_date, sales_amount from gold.fact_sales where order_date is not null""",
    engine,
)
df_sales["order_date"] = pd.to_datetime(df_sales["order_date"])
df_sales.info()
analysis_date = df_sales["order_date"].max()
snapshot_date = analysis_date + pd.Timedelta(days=1)
print("snapshot date: ", snapshot_date)

<class 'pandas.DataFrame'>
RangeIndex: 60379 entries, 0 to 60378
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype        
---  ------        --------------  -----        
 0   customer_key  60379 non-null  int64        
 1   order_number  60379 non-null  str          
 2   order_date    60379 non-null  datetime64[s]
 3   sales_amount  60379 non-null  int64        
dtypes: datetime64[s](1), int64(2), str(1)
memory usage: 1.8 MB
snapshot date:  2014-01-29 00:00:00


## 3. Raw RFM values per customer

In [27]:
rfm = (
    df_sales.groupby("customer_key")
    .agg(
        recency_days=("order_date", lambda s: (snapshot_date - s.max()).days),
        frequency=("order_number", "nunique"),
        monetary=("sales_amount", "sum"),
    )
    .sort_values("monetary")
    .reset_index()
)
rfm["avg_order_value"] = rfm["monetary"] / rfm["frequency"].replace(0, np.nan)
print(rfm.describe())

       customer_key  recency_days     frequency      monetary  avg_order_value
count  18482.000000  18482.000000  18482.000000  18482.000000     18482.000000
mean    9242.344281    190.335786      1.496429   1588.099665       911.751114
std     5335.992085    146.296278      1.101186   2123.778423      1040.667915
min        1.000000      1.000000      1.000000      2.000000         2.000000
25%     4621.250000     87.000000      1.000000     50.000000        40.000000
50%     9242.500000    169.000000      1.000000    272.000000       157.000000
75%    13862.750000    264.000000      2.000000   2511.000000      1770.000000
max    18484.000000   1127.000000     28.000000  13294.000000      3578.000000


18,482 customers. Median recency is 169 days. Most customers ordered once (the 75th percentile of frequency is 2 orders), and spend is right-skewed (median 272 vs. mean 1,588).

## 4. RFM scores (1-4)

Each metric is ranked first and then cut into four equal-sized groups with `qcut`. Ranking before cutting avoids duplicate bin edges, because most customers have only one order. Recency is reversed so that the most recent customers get the highest score.

In [35]:
rfm["r"] = pd.qcut(
    rfm["recency_days"].rank(method="first"), 4, labels=[4, 3, 2, 1]
).astype(int)
rfm["f"] = pd.qcut(
    rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]
).astype(int)
rfm["m"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(
    int
)

rfm["rfm_score"] = rfm[["r", "f", "m"]].sum(axis=1)
rfm["rfm_code"] = rfm["r"].astype(str) + rfm["f"].astype(str) + rfm["m"].astype(str)
print(rfm[["r", "f", "m", "rfm_score"]].describe())

                  r             f             m     rfm_score
count  18482.000000  18482.000000  18482.000000  18482.000000
mean       2.500000      2.500000      2.500000      7.500000
std        1.118113      1.118113      1.118113      2.471465
min        1.000000      1.000000      1.000000      3.000000
25%        1.250000      1.250000      1.250000      6.000000
50%        2.500000      2.500000      2.500000      8.000000
75%        3.750000      3.750000      3.750000      9.000000
max        4.000000      4.000000      4.000000     12.000000


## 5. Business-friendly segment labels

The rules are checked from top to bottom and the first match wins:

| Segment | Rule |
|---|---|
| champions | R ≥ 3, F ≥ 3, M ≥ 3 |
| at risk - high value | R ≤ 2, M ≥ 3 |
| loyal/promising | R ≥ 3, F ≥ 2, M ≥ 2 |
| loyal but inactive | F ≥ 3, M ≥ 2 |
| hibernating / low value | everything else |

In [37]:
def label_segment(row):
    r, f, m = row["r"], row["f"], row["m"]

    if r >= 3 and f >= 3 and m >= 3:
        return "champions"
    elif r <= 2 and m >= 3:
        return "at risk- high value"
    elif r >= 3 and f >= 2 and m >= 2:
        return "loyal/promising"
    elif f >= 3 and m >= 2:
        return "loyal but inactive"
    else:
        return "hibernating / low value"


rfm["segment_label"] = rfm.apply(label_segment, axis=1)
print(rfm["segment_label"].value_counts())

segment_label
hibernating / low value    6402
at risk- high value        4724
champions                  3692
loyal/promising            3262
loyal but inactive          402
Name: count, dtype: int64


Segment sizes: hibernating / low value 6,402 · at risk - high value 4,724 · champions 3,692 · loyal/promising 3,262 · loyal but inactive 402.

## 6. Segment summary

In [42]:
summary = (
    rfm.groupby("segment_label")
    .agg(
        customers=("customer_key", "count"),
        total_revenue=("monetary", "sum"),
        avg_revenue=("monetary", "mean"),
        median_revenue=("monetary", "median"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)
summary["revenue_share_%"] = (
    100 * summary["total_revenue"] / summary["total_revenue"].sum()
).round(1)
summary["customer_share_%"] = (
    100 * summary["customers"] / summary["customers"].sum()
).round(1)
summary["revenue_per_customer"] = (
    summary["total_revenue"] / summary["customers"]
).round(2)

print(summary)

                         customers  total_revenue  avg_revenue  \
segment_label                                                    
at risk- high value           4724       14486780  3066.634208   
champions                     3692       13708917  3713.141116   
loyal/promising               3262         839089   257.231453   
hibernating / low value       6402         270149    42.197595   
loyal but inactive             402          46323   115.231343   

                         median_revenue  avg_recency_days  avg_frequency  \
segment_label                                                              
at risk- high value              2511.5        319.907917       1.574301   
champions                        3181.5         92.062568       2.148971   
loyal/promising                   115.0         84.042918       1.474249   
hibernating / low value            36.0        202.901281       1.033583   
loyal but inactive                104.5        232.646766       2.139303   

    

Champions and at-risk high-value customers make up 45.6% of customers but generate about 96% of revenue (46.7% and 49.4%). Hibernating customers are 34.6% of customers and 0.9% of revenue.

## 7. Recommended action per segment

In [44]:
action_map = {
    "champions": "Retain, loyalty benefits, cross-sell / premium offers",
    "at risk- high value ": "Retention campaign, service recovery, targeted win-back",
    "loyal/promising": "Increase frequency and basket size",
    "Loyal but Inactive": "Reactivation campaign",
    "hibernating / low value ": "Low-cost automated re-engagement",
}
summary["recommended_action"] = summary.index.to_series().map(action_map)
print(
    summary[
        ["customers", "revenue_share_%", "revenue_per_customer", "recommended_action"]
    ]
)

                         customers  revenue_share_%  revenue_per_customer  \
segment_label                                                               
at risk- high value           4724             49.4               3066.63   
champions                     3692             46.7               3713.14   
loyal/promising               3262              2.9                257.23   
hibernating / low value       6402              0.9                 42.20   
loyal but inactive             402              0.2                115.23   

                                                        recommended_action  
segment_label                                                               
at risk- high value                                                    NaN  
champions                Retain, loyalty benefits, cross-sell / premium...  
loyal/promising                         Increase frequency and basket size  
hibernating / low value                                                NaN 

## 8. Charts

In [61]:
order = summary.sort_values("total_revenue").index
plt.figure(figsize=(8, 5))
sns.barplot(data=summary.loc[order].reset_index(), x="total_revenue", y="segment_label")
plt.title("Revenue by RFM Segment")
plt.xlabel("Total Revenue")
plt.tight_layout()
plt.savefig("output/04_rfm_revenue_by_segment_v2.png", dpi=120)
plt.close()


x = np.arange(len(summary))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width / 2, summary["customer_share_%"], width, label="Customer Share %")
ax.bar(x + width / 2, summary["revenue_share_%"], width, label="Revenue Share %")
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=20, ha="right")
ax.legend()
ax.set_title("Customer Share vs Revenue Share")
plt.tight_layout()
plt.savefig("output/05_rfm_customer_vs_revenue_share_v2.png", dpi=120)
plt.close()

![Revenue by RFM segment](output/04_rfm_revenue_by_segment_v2.png)

![Customer share vs revenue share](output/05_rfm_customer_vs_revenue_share_v2.png)

## 9. Save outputs

In [62]:
rfm.to_csv("output/rfm_segments_v2.csv", index=False)
summary.to_csv("output/rfm_segment_summary_v2.csv")
print("RFM analysis complete.")

RFM analysis complete.
